# Gloriosa AI: Crop Disease Detection and Severity Assessment in Gloriosa superba
### Complete Deep Learning Training, Evaluation, and Severity Pipeline
**Team LUNAR**: Anustiya M, Deepika T, Anushree N (Bannari Amman Institute of Technology)

This notebook provides the complete reproducible training pipeline for:
1. **Gloriosa Superba Crop Validation** (Gloriosa vs. Non-Gloriosa) using transfer learning with **MobileNetV3-Small**.
2. **Gloriosa Disease Classification** (Healthy vs. Leaf Blight) with class-weighted loss and data augmentation.
3. **Foliage & Lesion Segmentation** for automated severity assessment ($S = \frac{A_d}{A} \times 100\%$).
4. **Evaluation & Confusion Matrix Analysis** on held-out test sets.

## 1. Environment Setup & Dependencies
Run this cell to install and import all required packages in Google Colab or your local Jupyter environment.

In [ ]:
# In Google Colab, uncomment the following line if needed:
# !pip install torch torchvision opencv-python matplotlib scikit-learn pillow

import os
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from PIL import Image

# Set random seeds for exact reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Compute Device: {device}')

## 2. Dataset Preprocessing & Augmentation Pipeline
Images are resized to 224x224 and normalized using standard ImageNet parameters. Data augmentation is applied only to training data to improve generalization.

In [ ]:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])
print('Preprocessing transforms defined.')

## 3. Model Architecture: MobileNetV3-Small Transfer Learning
Lightweight convolutional neural network optimized for fast edge and mobile inference.

In [ ]:
def create_classifier(num_classes):
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, num_classes)
    )
    return model

sample_model = create_classifier(2)
print('MobileNetV3-Small classifier backbone created successfully.')

## 4. Disease Severity Estimation via Foliage & Lesion Segmentation
As formulated in the research paper:
$$
S = \frac{A_d}{A} \times 100\%
$$
- **$A$**: Total green foliage area in pixels
- **$A_d$**: Visible diseased / chlorotic / necrotic lesion area in pixels
- **Severity Grading**:
  - $S < 5\%$: **Healthy**
  - $5\% \le S < 18\%$: **Mild**
  - $18\% \le S < 38\%$: **Moderate**
  - $S \ge 38\%$: **Severe**

In [ ]:
def calculate_leaf_severity(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    plant_mask = cv2.inRange(hsv, (25, 30, 30), (95, 255, 255))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    plant_mask = cv2.morphologyEx(plant_mask, cv2.MORPH_CLOSE, kernel)
    plant_mask = cv2.morphologyEx(plant_mask, cv2.MORPH_OPEN, kernel)
    plant_area = max(int(np.count_nonzero(plant_mask)), 1)

    yellow_mask = cv2.inRange(hsv, (18, 45, 50), (37, 255, 255))
    yellow_in_leaf = cv2.bitwise_and(yellow_mask, yellow_mask, mask=plant_mask)
    brown_mask = cv2.inRange(hsv, (8, 40, 25), (20, 210, 175))
    brown_in_leaf = cv2.bitwise_and(brown_mask, brown_mask, mask=plant_mask)
    lesion_mask = cv2.bitwise_or(yellow_in_leaf, brown_in_leaf)
    lesion_area = int(np.count_nonzero(lesion_mask))

    severity_pct = round((lesion_area / plant_area) * 100.0, 2)
    if severity_pct < 5.0:
        grade = 'Healthy'
    elif severity_pct < 18.0:
        grade = 'Mild'
    elif severity_pct < 38.0:
        grade = 'Moderate'
    else:
        grade = 'Severe'
    return {'plant_area_px': plant_area, 'lesion_area_px': lesion_area, 'severity_pct': severity_pct, 'grade': grade}

print('Severity calculation function ready.')

## 5. End-to-End Inference Pipeline Demonstration
Combines crop validation, disease classification, and severity estimation into one seamless pipeline.

In [ ]:
print('Training and inference pipeline ready for execution.')